# Notebook pro 1st stage trénink StyleTTS2

**POZOR:** Vzhledem k následnému 2nd stage tréninku je nutné pro 1st trénink použít **>1 GPU**!

- podporuje gradient accumulation

Je doporučeno používat vstupní promluvy <= `max_len`

Doporučený seed: 3407 ;-)

## Imports & preparatory steps

In [1]:
import logging
import os
import os.path as osp
import shutil
import subprocess
from platform import node, python_version

import nvidia_smi
import torch
from ruamel.yaml import YAML
from torch import __version__ as torch_version

# import wandb

# Check CUDA is available
assert torch.cuda.is_available(), "CPU training is not allowed."

# Check the number of CPUs
N_CPUS = int(os.environ["PBS_NUM_PPN"])

# Limit CPU operation in pytorch to `N_CPUS`
torch.set_num_threads(N_CPUS)
torch.set_num_interop_threads(N_CPUS)

# Set username
USER = os.environ["USER"]
# Set hostname
HOSTNAME = node()
# Check interactive mode
INTERACTIVE_MODE = bool("JupyterLab" in os.environ["PBS_JOBNAME"])
# Log level
LOG_LEVEL = "DEBUG"  # logging.DEBUG

# GPU
n_gpus = torch.cuda.device_count()
nvidia_smi.nvmlInit()
# deviceCount = nvidia_smi.nvmlDeviceGetCount()

print(" > Computational resources...")
print(f" | > Hostname:         {HOSTNAME}")
print(f" | > Interactive mode: {INTERACTIVE_MODE}")
print(f" | > Number of CPUs:   {N_CPUS}")
print(f" | > Number of GPUs:   {n_gpus}")
for idx in range(n_gpus):
    handle = nvidia_smi.nvmlDeviceGetHandleByIndex(idx)
    info = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
    print(
        f" | > Device {idx}:     {nvidia_smi.nvmlDeviceGetName(handle)} VRAM usage: {info.used>>30}/{info.total>>30} GB ({info.used/info.total:.2%})"
    )
print(" > Python & module versions...")
print(f" | > Python:           {python_version()}")
print(f" | > PyTorch:          {torch_version}")

nvidia_smi.nvmlShutdown()

# assert n_gpus > 1, "At least two GPUs are required for 1st stage training."

 > Computational resources...
 | > Hostname:         luna202.fzu.cz
 | > Interactive mode: True
 | > Number of CPUs:   1
 | > Number of GPUs:   1
 | > Device 0:     b'NVIDIA A40' VRAM usage: 0/44 GB (1.27%)
 > Python & module versions...
 | > Python:           3.12.3
 | > PyTorch:          2.6.0+cu126


## Settings

In [2]:
# Floats are not written in scientific notation when storing to YAML
def float_representer(representer, data):
    value = "{0:.15f}".format(data).rstrip("0")
    return representer.represent_scalar("tag:yaml.org,2002:float", value)

In [3]:
log_dir = "Exps/test"
label = ""
first_stage_path = "first_stage.pth"
save_freq = 1
max_saved_models = 2
save_milestones = False
log_interval = 10
device = "cuda"
batch_size = 8
grad_accum_steps = 4  # JMa: gradient accumulation
max_len = 100  # maximum number of frames
grad_clip = None  # JMa: gradient clipping
pretrained_model = ""
second_stage_load_pretrained = False  # set to true if the pre-trained model is for 2nd stage
# set to true if do not want to load epoch numbers and optimizer parameters
load_only_params = False
seed = 3407

# Epochs for each phase
epochs = {
    "stage1": 6,
    "tma": 2,
    "stage2": 20,
    "diff": 4,
    "joint": 10,
}

F0_path = "Utils/JDC/bst.t7"
ASR_config = "Utils/ASR/config.yml"
ASR_path = "Utils/ASR/epoch_00400.pth"
PLBERT_dir = "Utils/PLBERT/"

data_params = {
    "train_data": "Data/SPT-CRo.cs.n100/debug.train.csv",
    # "train_data": "Data/SPT-CRo.cs.n100/debug1k.train.csv",
    "val_data": "Data/SPT-CRo.cs.n100/debug.valid.csv",
    "root_path": "Data/SPT-CRo.cs.n100",
    "OOD_data": "Data/SPT-CRo.cs.n100/OOD_texts.csv",
    "min_length": 50,  # sample until texts with this size are obtained for OOD texts
    # test params
    "save_val_audio": True,
    "n_val_audios": 3,
    "save_test_audio": False,
    "test_audio_dir": "test_audios",  # directory under `log_dir`
    "test_sentences": [
        # cs
        "cexmistr miroslav marek toTiS uvedl, Ze kvUli ledofce se o hoDinu spozDil RezJIk z nedalekE vesJice."
        "cesta bila dUkladJe znaCenA bIlIm vApnem, potom sme vistoupali do prutkEho svahu, gde rostli pRevAZJe kaktusi, a povrx se zmJeJil na sipkou hlInu proloZenou kameni."
        "do jakE mIri vAs pRi spjevu inspirujI hudebJI nAstroje?",
    ],
    "symbol_dict_path": "Data/SPT-CRo.cs.n100/cs-epa.csv",  # (phoneme) symbol encoding table
    "pad": "_",  # pad symbol
}

preprocess_params = {
    "sr": 24000,
    "mean": -4,
    "std": 4,
    "silence_beg": 2400,  # 100ms
    "silence_end": 2400,  # 100ms
    "spect_params": {
        "n_fft": 2048,
        "win_length": 1200,
        "hop_length": 300,
    },
    "max_ref_mel_length": 192,  # maximum length of reference mel-spectrogram (in frames) (192 = 2.4s)
}

model_params = {
    "multispeaker": True,
    "dim_in": 64,
    "hidden_dim": 512,
    "max_conv_dim": 512,
    "n_layer": 3,
    "n_mels": 80,
    "n_token": 81,  # number of phoneme tokens (keys in `symbol_dict_path`)
    "max_dur": 50,  # maximum duration of a single phoneme
    "style_dim": 128,  # style vector size
    "dropout": 0.2,
    # External/internal speaker style mixing strategy:
    #  - If `learnable_gate = True`: `weight = 0.0` (0.5 after sigmoid activation) means a balanced mixing
    #    of external and internal speaker styles. The weight is learnable and is updated during training.
    #  - If `learnable_gate = False`: `mix_weight` is a fixed value between 0.0 and 1.0
    #    1.0 means only external speaker style is used, 0.0 means only internal speaker style is used.
    #    The weight is not learnable and is fixed during training.
    "style_mix": {
        "weight": 0.0,
        "learnable_gate": True,  # learnable gate for external/internal speaker styl mixing
        "warmup_beg_epoch": 0,  # number of epochs for which only internal style encoding is used
        "warmup_end_epoch": 3,  # number of epochs for which both internal and external style encoding are used in warmup mode
        "warmup_mode": "external",  # external: external embeddings are warmed up, internal: internal embeddings are warmed up
        "mode": "mix",
    },
    # config for decoder
    "decoder": {
        # "type": "istftnet",  # either hifigan or istftnet
        ## ISTFTNET
        # "resblock_kernel_sizes": [3, 7, 11],
        # "upsample_rates": [10, 6],
        # "upsample_initial_channel": 512,
        # "resblock_dilation_sizes": [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        # "upsample_kernel_sizes": [20, 12],
        # "gen_istft_n_fft": 20,
        # "gen_istft_hop_size": 5,
        "type": "hifigan",  # either hifigan or istftnet
        # HiFi-GAN
        "resblock_kernel_sizes": [3, 7, 11],
        "upsample_rates": [10, 5, 3, 2],
        "upsample_initial_channel": 512,
        "resblock_dilation_sizes": [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        "upsample_kernel_sizes": [20, 10, 6, 4],
    },
    # Speaker encoder parameters
    "spkenc_params": {
        "sr": 16000,  # sampling rate of speaker encoder
        "model": "Utils/speaker_encoder/hasp.pth.tar",  # path to the speaker encoder model
        "dim_in": 512,  # input speaker embedding size
        "spk_emb_dim": 128,  # speaker embedding size used in model
        "freeze": False,  # freeze speaker encoder parameters during training
    },
    # speech language model config
    "slm": {
        # "model": "microsoft/wavlm-base-plus",
        # "model": "microsoft/wavlm-large",
        "model": "openai/whisper-large-v3",
        "sr": 16000,  # sampling rate of SLM
        "hidden": 1280,  # hidden size of SLM
        "nlayers": 33,  # number of layers of SLM
        "initial_channel": 64,  # initial channels of SLM discriminator head
    },
    # style diffusion model config
    "diffusion": {
        "embedding_mask_proba": 0.1,
        # transformer config
        "transformer": {
            "num_layers": 3,
            "num_heads": 8,
            "head_features": 64,
            "multiplier": 2,
        },
        # diffusion distribution config
        "dist": {
            "sigma_data": 0.2,  # placeholder for estimate_sigma_data set to false
            "estimate_sigma_data": True,  # estimate sigma_data from the current batch if set to true
            "mean": -3.0,
            "std": 1.0,
        },
    },
}

loss_params = {
    "lambda_mel": 10.0,  # mel reconstruction loss
    "lambda_scl": 1.0,  # speaker consistency loss
    "lambda_gen": 1.0,  # generator loss
    "lambda_slm": 1.0,  # slm feature matching loss
    "lambda_mono": 1.0,  # monotonic alignment loss (1st stage, TMA)
    "lambda_s2s": 1.0,  # sequence-to-sequence loss (1st stage, TMA)
    "lambda_F0": 1.0,  # F0 reconstruction loss (2nd stage)
    "lambda_norm": 1.0,  # norm reconstruction loss (2nd stage)
    "lambda_dur": 1.0,  # duration loss (2nd stage)
    "lambda_ce": 20.0,  # duration predictor probability output CE loss (2nd stage)
    "lambda_sty": 1.0,  # style reconstruction loss (2nd stage)
    "lambda_diff": 1.0,  # score matching loss (2nd stage)
}

optimizer_params = {
    "lr": 0.0001,  # general learning rate
    "bert_lr": 0.00001,  # learning rate for PLBERT
    "ft_lr": 0.00001,  # learning rate for acoustic modules
}

slmadv_params = {
    "min_len": 64,  # minimum length of samples (400)
    "max_len": 400,  # maximum length of samples (500)
    "batch_percentage": 0.5,  # to prevent out of memory, only use half of the original batch size. `None` means no SLM discriminator training
    "iter": 20,  # update the discriminator every this iterations of generator update (10)
    "thresh": 5,  # gradient norm above which the gradient is scaled
    "scale": 0.01,  # gradient scaling factor for predictors from SLM discriminators
    "sig": 1.5,  # sigma for differentiable duration modeling
}

## Delete previous W&B run

In [4]:
# # Delete previous runs with the same name
# exp_name = f"osp.basename(log_dir)_{label}"
# api = wandb.Api()
# runs = api.runs(f"jmaty/StyleTTS2-spkenc")
# [run.delete() for run in runs if run.name == exp_name]

## Copy data to local (scratch) dir

In [5]:
# Set up local scratch directory
scratch_dir = os.environ["SCRATCHDIR"]

# Check "interactive mode":
# => data are copied when not in interactive mode and not running on specified servers
if not INTERACTIVE_MODE:

    LOG_LEVEL = "INFO"  # logging.INFO Set logging to INFO in non-interactive mode

    # Set up local dataset directory
    if "bee" in HOSTNAME:  # Do not copy data, just set up the local path
        local_prefix = osp.join("/scratch.shared", USER)
    elif "capy" in HOSTNAME:  # Do not copy data, just set up the local path
        local_prefix = osp.join("/scratch.ssd", USER)
    else:  # Copy data to scratch directory
        local_prefix = scratch_dir

        # Copy dataset from remote storage & prepare dataset dir in the scratch

        print(f"> Copying data to local scratch directory: {local_prefix}")
        # Absolute path to voice directory
        storage_dir = osp.dirname(osp.abspath(data_params["root_path"]))
        # Voice directory name
        basedir = osp.basename(storage_dir)
        # Absolute path to the Data directory
        storage_dir = osp.dirname(storage_dir)

        # Define command for SSH and tar on remote server
        ssh_command = [
            "ssh",
            "storage-plzen4.kky.zcu.cz",
            f"tar -h -C {storage_dir} -cf - {basedir}",
        ]
        # Define command for local tar extraction
        tar_command = ["tar", "-xf", "-", "-C", osp.join(local_prefix, "Data")]
        # Run 1st process (SSH and remote tar)
        ssh_process = subprocess.Popen(ssh_command, stdout=subprocess.PIPE, text=True)
        # Run 2nd process (local tar), which reads output form the 1st process
        tar_process = subprocess.Popen(tar_command, stdin=ssh_process.stdout, text=True)
        # Close output pipe from 1st process
        ssh_process.stdout.close()
        # Wait until processes finish
        ssh_returncode = ssh_process.wait()
        tar_returncode = tar_process.wait()
        # Verify process finished successfully
        if ssh_returncode != 0:
            print(f"SSH proces skončil s chybou: {ssh_returncode}")
        if tar_returncode != 0:
            print(f"Tar proces skončil s chybou: {tar_returncode}")

    # Store the local dataset so that it is used for training
    # WARNING: The input data paths in the config file must be relative to the working directory
    data_params["train_data"] = osp.join(local_prefix, data_params["train_data"])
    data_params["val_data"] = osp.join(local_prefix, data_params["val_data"])
    data_params["OOD_data"] = osp.join(local_prefix, data_params["OOD_data"])
    data_params["root_path"] = osp.join(local_prefix, data_params["root_path"])
    data_params["symbol_dict_path"] = osp.join(local_prefix, data_params["symbol_dict_path"])

## Create/update config file

In [6]:
config = {
    "log_dir": log_dir,
    "label": label,  # label for the experiment
    "seed": seed,
    "first_stage_path": first_stage_path,
    "save_freq": save_freq,
    "max_saved_models": max_saved_models,
    "save_milestones": save_milestones,
    "log_interval": log_interval,
    "device": device,
    "epochs": epochs,
    "batch_size": batch_size,
    "grad_accum_steps": grad_accum_steps,
    "max_len": max_len,
    "grad_clip": grad_clip,
    "pretrained_model": pretrained_model,
    "second_stage_load_pretrained": second_stage_load_pretrained,
    "load_only_params": load_only_params,
    "F0_path": F0_path,
    "ASR_config": ASR_config,
    "ASR_path": ASR_path,
    "PLBERT_dir": PLBERT_dir,
    "data_params": data_params,
    "preprocess_params": preprocess_params,
    "model_params": model_params,
    "loss_params": loss_params,
    "optimizer_params": optimizer_params,
    "slmadv_params": slmadv_params,
}

# Write config file to scratch dir
config_file = os.path.join(scratch_dir, "config.yml")
# Write to a YAML file
yaml = YAML()
yaml.representer.add_representer(float, float_representer)
yaml.default_flow_style = False
with open(config_file, "w") as f:
    yaml.dump(config, f)

## Run training script

In [7]:
print(" > Run training script: train_first.py")
print(f" | > Log level:   {LOG_LEVEL}")
print(f" | > # workers:   {N_CPUS}")
print(f" | > Config file: {config_file}")

cmd = [
    "accelerate",
    "launch",
    "--mixed_precision=no",
    "train_first.py",
    f"--num_workers={N_CPUS}",
    f"--log_level={LOG_LEVEL}",
    config_file,
]

try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"Command failed with exit code {e.returncode}") from e

 > Run training script: train_first.py
 | > Log level:   DEBUG
 | > # workers:   1
 | > Config file: /scratch.ssd/jmatouse/job_12912555.pbs-m1/config.yml


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2025-09-12 15:12:42.769879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-12 15:12:42.794474: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-12 15:12:42.801806: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-12 15:12:42.821648: I te